# going back to 1D DSM!

Nobuaki Fuji 29/07/2026

The challenge is now to get the SH and PSV equation in spherical harmonics symbolically

In [1]:
# Locate flexOPT securely without relying on @__DIR__ (unreliable in IJulia).
# If this notebook is outside the repository, set ENV["FLEXOPT_ROOT"] first.
import Pkg
function find_flexopt_root(start_dir=pwd())
    candidates = String[]
    if haskey(ENV, "FLEXOPT_ROOT")
        push!(candidates, abspath(expanduser(ENV["FLEXOPT_ROOT"])))
    end
    directory = abspath(start_dir)
    while true
        push!(candidates, directory)
        parent = dirname(directory)
        parent == directory && break
        directory = parent
    end
    for candidate in unique(candidates)
        project_file = joinpath(candidate, "Project.toml")
        source_dir = joinpath(candidate, "src")
        if isfile(project_file) && isfile(joinpath(source_dir, "commonBatchs.jl"))
            return candidate
        end
    end
    error("Cannot locate flexOPT. Start Jupyter inside the repository or set ENV[\"FLEXOPT_ROOT\"] to its absolute path.")
end

flexopt_root = find_flexopt_root()
Pkg.activate(flexopt_root)
@show VERSION Threads.nthreads() Base.active_project()

# Metal must be loaded before batchGPU.jl selects the backend.
using Metal
Metal.functional() || error("Metal.jl is loaded, but cannot access the Apple GPU")
@show Metal.devices()

include(joinpath(flexopt_root, "src", "batchFiles", "batchGPU.jl"))
include(joinpath(flexopt_root, "src", "commonBatchs.jl"))
include(joinpath(flexopt_root, "src", "planet1D.jl"))
planet1D.configure_input!()
include(joinpath(flexopt_root, "src", "GeoPoints.jl"))
using .commonBatchs, .planet1D, .GeoPoints

include(joinpath(flexopt_root, "src", "flexOPT.jl"))
using .flexOPT


  Activating 

VERSION = v"1.12.6"

project at `~/Documents/Github/flexOPT`



Threads.nthreads() = 1
Base.active_project() = "/Users/nobuaki/Documents/Github/flexOPT/Project.toml"
Metal.devices() = Metal.MTL.MTLDeviceInstance[Metal.MTL.MTLDeviceInstance (object of type AGXG13XDevice)]
devs = Metal.devices() = Metal.MTL.MTLDeviceInstance[Metal.MTL.MTLDeviceInstance (object of type AGXG13XDevice)]
→ Using Metal backend (1 device(s))
Selected backend type: MetalBackend


In [ ]:
# we make our 'famous equations'
using Symbolics, LinearAlgebra
@variables r θ ϕ
@variables ω
@variables ρ(r) (C(r))[1:3,1:3,1:3,1:3] u(r,θ,ϕ)[1:3] (M(r,θ,ϕ))[1:3,1:3]
@variables λ(r) μ(r)
    #@variables ρ C[1:3,1:3,1:3,1:3] u(x,y,z,t)[1:3] f1,f2,f3 #(M(x,y,z))[1:3,1:3]
    #@variables λ μ
δ=Matrix(I, 3, 3) # Kronecker's delta
@tullio C[i,j,k,l] := λ * δ[i,j]*δ[k,l]+μ*(δ[i,k]*δ[j,l]+δ[i,l]*δ[j,k])
@tullio traction[i] := ∇₃[j](C[i,j,k,l]*∇₃[l](u[k])) # -> it should be rewritten for spherical harmonics! 
@tullio derivMoment[i] := ∇₃[j](M[i,j])

exprs = ρ* ω^2 * (u[1]) - traction[1], ρ* ∂t²(u[2]) - traction[2], ρ* ∂t²(u[3]) - traction[3]
fields=u[1], u[2], u[3]
vars = ρ, λ, μ

#extexprs = derivMoment[1], derivMoment[2], derivMoment[3]
#extfields = M[1,1], M[1,2], M[1,3], M[2,2], M[2,3],M[3,3]
#extvars = M[1,1], M[1,2], M[1,3], M[2,2], M[2,3],M[3,3]
    
coordinates =(r, θ, ϕ)

UndefVarError: UndefVarError: `I` not defined in `Main`
Suggestion: check for spelling errors or missing imports.
Hint: a global variable of this name also exists in LinearAlgebra.